In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
import plotly.graph_objects as go
#import plotly.express as px
from plotly_resampler import FigureResampler, register_plotly_resampler, FigureWidgetResampler
from datetime import datetime
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler
import random
scaler = MinMaxScaler()
fig = go.Figure()

In [12]:
def generate_sincurve(year, amplitude=1.0, frequency=1.0):
    start_date = datetime(year, 1, 1, 0, 0)
    end_date = datetime(year + 1, 1, 1, 0, 0)
    total_minutes = int((end_date - start_date).total_seconds() / 24)
    timestamps = [start_date + timedelta(minutes=i) for i in range(total_minutes)]
    sin_values = amplitude * np.sin(2 * np.pi * frequency * np.arange(total_minutes) / (24 * 60))

    df = pd.DataFrame({
        'tampstamp': timestamps,
        'values': sin_values
    })
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    return df

def generate_sincurve_with_anomalies(year, amplitude=1.0, frequency=1.0, anomaly_fraction=0.00005, anomaly_magnitude=7.0):
    start_date = datetime(year, 1, 1, 0, 0)
    end_date = datetime(year + 1, 1, 1, 0, 0)
    total_intervals = int((end_date - start_date).total_seconds() / 15)
    timestamps = [start_date + timedelta(seconds=15*i) for i in range(total_intervals)]
    sin_values = amplitude * np.sin(2 * np.pi * frequency * np.arange(total_intervals) / (24 * 3600 / 15))

    # Introduce anomalies
    num_anomalies = int(total_intervals * anomaly_fraction)
    anomaly_indices = random.sample(range(total_intervals), num_anomalies)
    for idx in anomaly_indices:
        sin_values[idx] += anomaly_magnitude * (random.random() - 0.5) * 2  # Randomly add/subtract anomaly_magnitude

    df = pd.DataFrame({
        'timestamp': timestamps,
        'values': sin_values
    })
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    return df

In [13]:
tmp = generate_sincurve_with_anomalies(2023)
print(len(tmp))
time_series_data = tmp['values'].values
time_series_data_normalized = scaler.fit_transform(time_series_data.reshape(-1, 1))

def create_dataset(data, time_step=1):
    X = []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), 0])
    return np.array(X)

time_step = 10
X = create_dataset(time_series_data_normalized, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

2102400


In [18]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, input_shape=(time_step, 1), return_sequences=True),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.RepeatVector(time_step),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# Train the model
history = model.fit(X, X, epochs=1, batch_size=128, validation_split=0.2, shuffle=False)

X_pred = model.predict(X)
mse = np.mean(np.power(X.reshape(X.shape[0], time_step) - X_pred.reshape(X_pred.shape[0], time_step), 2), axis=1)

# Threshold for anomaly detection
threshold = np.percentile(mse, 99.9)

/Users/jacksonwelch/Documents/yaf-docker/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 10, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_2 (RepeatVector)  │ (None, 10, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 10, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 10, 1)          │            33 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,665 (147.13 KB)

 Trainable params: 37,665 (147.13 KB)

 Non-trainable params: 0 (0.00 B)

13140/13140 ━━━━━━━━━━━━━━━━━━━━ 192s 15ms/step - loss: 0.0030 - val_loss: 9.4946e-04
65700/65700 ━━━━━━━━━━━━━━━━━━━━ 102s 2ms/step


In [20]:
anomalies = mse > threshold
print(mse)
print(f'Threshold: {threshold}')
anomalies_index = np.where(anomalies)[0]

print(f'Number of anomalies detected: {len(anomalies_index)}')
print(f'Indices of anomalies: {anomalies_index}')
fig = go.Figure()
register_plotly_resampler(mode="auto", default_n_shown_samples=1500)

# Plot anomalies
fig.add_trace(go.Scatter(
    x=tmp['timestamp'],
    y=tmp['values'],
    mode='lines',
    name='Time Series Data'
))

# Add the anomalies
fig.add_trace(go.Scatter(
    x=tmp['timestamp'].iloc[anomalies_index],
    y=tmp['values'].iloc[anomalies_index],
    mode='markers',
    marker=dict(color='red', size=5),
    name='Anomalies'
))
fig = FigureWidgetResampler(fig)
display(fig)

[0.00094589 0.00094684 0.00094779 ... 0.00093347 0.00093443 0.00093539]
Threshold: 0.002202760690596593
Number of anomalies detected: 2066
Indices of anomalies: [   1432    1433    1434 ... 2098073 2098074 2098075]


FigureWidgetResampler({
    'data': [{'mode': 'lines',
              'name': ('<b style="color:sandybrown">[R' ... ' style="color:#fc9944">~9h</i>'),
              'type': 'scatter',
              'uid': 'd4ab1c1e-af29-46f6-93c2-0b3353c940e1',
              'x': array([datetime.datetime(2023, 1, 1, 0, 0),
                          datetime.datetime(2023, 1, 1, 6, 0),
                          datetime.datetime(2023, 1, 1, 17, 31, 15), ...,
                          datetime.datetime(2023, 12, 31, 6, 28, 30),
                          datetime.datetime(2023, 12, 31, 18, 0),
                          datetime.datetime(2023, 12, 31, 23, 59, 45)], dtype=object),
              'y': array([ 0.        ,  1.        , -0.99214202, ...,  0.99227791, -1.        ,
                          -0.00109083])},
             {'marker': {'color': 'red', 'size': 5},
              'mode': 'markers',
              'name': '<b style="color:sandybrown">[R]</b> Anomalies <i style="color:#fc9944">~9h</i>',
     

In [ ]:
anomalies_index

In [4]:

fig.add_trace(go.Scatter(
    x=tmp['timestamp'],
    y=tmp['values'],
    mode='lines',
    name='Sin Curve'
))

fig.update_layout(
    title='Sin Curve',
    xaxis_title='Timestamp',
    yaxis_title='Values'
)

display(fig)

# fig.show()

NameError: name 'fig' is not defined